In [ ]:
import math 
import numpy as np
import pandas as pd
from tqdm import tqdm
from quantum_close_neighbors.grovers import FastAbstractGrovers
from quantum_close_neighbors.close_neighbors import close_neighbors, CloseNeighborsHyperparameters, DEFAULT_HYPERPARAMETERS


Experiments in tuning hyperparameters of close neighbors 1 (with replacement)

### Experiment 1
* Default hyperparameters
* Modify alpha
* Compare theoretical success rate to the actual success rate

In [ ]:
n_particles = 100
n_neighbors = 30

nu = math.comb(n_particles, 2)          # nu is n choose 2
mu = n_particles * n_neighbors // 2     # mu is the number of close neighbor

theoretical_success_rates = [0.99, 0.9, 0.75, 0.5, 0.25]
alphas = []

actual_success_rates = []
average_iterations = []
average_iterations_precritical = []
average_iterations_postcritical = []
average_iterations_final_collection = []
average_pairs_found = []

runs = 100


hyperparameters = DEFAULT_HYPERPARAMETERS

for i, theoretical_success_rate in enumerate(theoretical_success_rates):

    # DESIRED_SUCCESS_RATE = 1 - n^(-alpha)
    # n^(-alpha) = 1 - DESIRED_SUCCESS_RATE
    # -alpha * log(n) = log(1 - DESIRED_SUCCESS_RATE)
    # alpha = log(1 - DESIRED_SUCCESS_RATE) / log(n)
    alpha = -math.log(1 - theoretical_success_rate) / math.log(nu)
    alphas.append(alpha)

    grover_iterations_precritical = []
    grover_iterations_postcritical = []
    grover_iterations_final_collection = []
    grover_iterations = []
    pairs_found = []
    successes = 0
    for _ in tqdm(range(runs)):
        n_found, iterations = close_neighbors(mu, alpha, hyperparameters, FastAbstractGrovers(nu, mu))
        pairs_found.append(n_found/mu)
        grover_iterations_precritical.append(iterations[0])
        grover_iterations_postcritical.append(iterations[1])
        grover_iterations_final_collection.append(iterations[2])
        grover_iterations.append(sum(iterations))
        if n_found == mu:
            successes += 1

    actual_success_rate = successes / runs
    actual_success_rates.append(actual_success_rate)
    average_iterations.append(np.mean(grover_iterations))
    average_iterations_precritical.append(np.mean(grover_iterations_precritical))
    average_iterations_postcritical.append(np.mean(grover_iterations_postcritical))
    average_iterations_final_collection.append(np.mean(grover_iterations_final_collection))
    average_pairs_found.append(np.mean(pairs_found))


# Make a table of the results with pandas
df = pd.DataFrame({
    "Theoretical Success Rate": theoretical_success_rates,
    "Alpha": alphas,
    "Actual Success Rate": actual_success_rates,
    "Average Pairs Found": average_pairs_found,
    "Average Iterations": average_iterations,
    "Average Precritical Iterations": average_iterations_precritical,
    "Average Postcritical Iterations": average_iterations_postcritical,
    "Average Final Collection Iterations": average_iterations_final_collection
})



df    

## Experiment 2
* Fix alpha at 1, all hyperparameters have default values
* When calculating the final number of iterations, try different constant factors (originally 4)
* In the original algorithm, (alpha+1) * 4 * A1 * log(n) iterations are used

In [ ]:
n_particles = 100
n_neighbors = 30

nu = math.comb(n_particles, 2)          # nu is n choose 2
mu = n_particles * n_neighbors // 2     # mu is the number of close neighbor

alpha = 1
final_collection_iteration_constants = [0.1, 0.5, 1, 2, 4] 

actual_success_rates = []
average_iterations = []
average_iterations_precritical = []
average_iterations_postcritical = []
average_iterations_final_collection = []
average_pairs_found = []

runs = 1000


for i, final_collection_constant in enumerate(final_collection_iteration_constants):


    hyperparameters = CloseNeighborsHyperparameters(
        final_collection_k=lambda n, alpha, A1 : (alpha+1) * final_collection_constant * A1 * np.log(n),
        m_factor_increase=1,
        c1=450,
        c2=6,
        lambda_val=6/5,
    )

    grover_iterations_precritical = []
    grover_iterations_postcritical = []
    grover_iterations_final_collection = []
    grover_iterations = []
    pairs_found = []
    successes = 0
    for _ in range(runs):
        n_found, iterations = close_neighbors(mu, alpha, hyperparameters, FastAbstractGrovers(nu, mu))
        pairs_found.append(n_found/mu)
        grover_iterations_precritical.append(iterations[0])
        grover_iterations_postcritical.append(iterations[1])
        grover_iterations_final_collection.append(iterations[2])
        grover_iterations.append(sum(iterations))
        if n_found == mu:
            successes += 1

    actual_success_rate = successes / runs
    actual_success_rates.append(actual_success_rate)
    average_iterations.append(np.mean(grover_iterations))
    average_iterations_precritical.append(np.mean(grover_iterations_precritical))
    average_iterations_postcritical.append(np.mean(grover_iterations_postcritical))
    average_iterations_final_collection.append(np.mean(grover_iterations_final_collection))
    average_pairs_found.append(np.mean(pairs_found))


# Make a table of the results with pandas
df = pd.DataFrame({
    "Final Collection Iteration Constant": final_collection_iteration_constants,
    "Success Rate": actual_success_rates,
    "Average Pairs Found": average_pairs_found,
    "Average Iterations": average_iterations,
    "Average Precritical Iterations": average_iterations_precritical,
    "Average Postcritical Iterations": average_iterations_postcritical,
    "Average Final Collection Iterations": average_iterations_final_collection
})


df    

## Experiment 3
* Fixing alpha at 1 and all hyperparameters at defaults
* After the precritical stage, we multiply m by a constant factor

In [ ]:
n_particles = 100
n_neighbors = 30

nu = math.comb(n_particles, 2)          # nu is n choose 2
mu = n_particles * n_neighbors // 2     # mu is the number of close neighbor

alpha = 1
m_factor_increases = list(np.arange(1, 1.25, 0.05))

actual_success_rates = []
average_iterations = []
average_iterations_precritical = []
average_iterations_postcritical = []
average_iterations_final_collection = []
average_pairs_found = []

runs = 200


for i, m_factor_increase in enumerate(m_factor_increases):


    hyperparameters = CloseNeighborsHyperparameters(
        final_collection_k=lambda n, alpha, A1 : (alpha+1) * 4 * A1 * np.log(n),
        m_factor_increase=m_factor_increase,
        c1=450,
        c2=6,
        lambda_val=6/5,
    )

    grover_iterations_precritical = []
    grover_iterations_postcritical = []
    grover_iterations_final_collection = []
    grover_iterations = []
    pairs_found = []
    successes = 0
    for _ in tqdm(range(runs)):
        n_found, iterations = close_neighbors(mu, alpha, hyperparameters, FastAbstractGrovers(nu, mu))
        pairs_found.append(n_found/mu)
        grover_iterations_precritical.append(iterations[0])
        grover_iterations_postcritical.append(iterations[1])
        grover_iterations_final_collection.append(iterations[2])
        grover_iterations.append(sum(iterations))
        if n_found == mu:
            successes += 1

    actual_success_rate = successes / runs
    actual_success_rates.append(actual_success_rate)
    average_iterations.append(np.mean(grover_iterations))
    average_iterations_precritical.append(np.mean(grover_iterations_precritical))
    average_iterations_postcritical.append(np.mean(grover_iterations_postcritical))
    average_iterations_final_collection.append(np.mean(grover_iterations_final_collection))
    average_pairs_found.append(np.mean(pairs_found))


# Make a table of the results with pandas
df = pd.DataFrame({
    "m Increase": m_factor_increases,
    "Success Rate": actual_success_rates,
    "Average Pairs Found": average_pairs_found,
    "Average Iterations": average_iterations,
    "Average Precritical Iterations": average_iterations_precritical,
    "Average Postcritical Iterations": average_iterations_postcritical,
    "Average Final Collection Iterations": average_iterations_final_collection
})


df    

## Experiment 4
* Modifying the c1 values which is the contant multiplied against alpha*log(n) in the paper
* In the paper it is 450 * alpha * log(n) and in theory lowering 450hlp but in practice it is negligable

In [ ]:
n_particles = 100
n_neighbors = 30

nu = math.comb(n_particles, 2)          # nu is n choose 2
mu = n_particles * n_neighbors // 2     # mu is the number of close neighbor

alpha = 1
c1_factor_increases = list(np.arange(5, 50, 10))

actual_success_rates = []
average_iterations = []
average_iterations_precritical = []
average_iterations_postcritical = []
average_iterations_final_collection = []
average_pairs_found = []

runs = 1000


for i, c1_values in enumerate(c1_factor_increases):


    hyperparameters = CloseNeighborsHyperparameters(
        final_collection_k=lambda n, alpha, A1 : (alpha+1) * 4 * A1 * np.log(n),
        m_factor_increase=1,
        c1=c1_values,
        c2=6,
        lambda_val=6/5,
    )

    grover_iterations_precritical = []
    grover_iterations_postcritical = []
    grover_iterations_final_collection = []
    grover_iterations = []
    pairs_found = []
    successes = 0
    for _ in tqdm(range(runs)):
        n_found, iterations = close_neighbors(mu, alpha, hyperparameters, FastAbstractGrovers(nu, mu))
        pairs_found.append(n_found/mu)
        grover_iterations_precritical.append(iterations[0])
        grover_iterations_postcritical.append(iterations[1])
        grover_iterations_final_collection.append(iterations[2])
        grover_iterations.append(sum(iterations))
        if n_found == mu:
            successes += 1

    actual_success_rate = successes / runs
    actual_success_rates.append(actual_success_rate)
    average_iterations.append(np.mean(grover_iterations))
    average_iterations_precritical.append(np.mean(grover_iterations_precritical))
    average_iterations_postcritical.append(np.mean(grover_iterations_postcritical))
    average_iterations_final_collection.append(np.mean(grover_iterations_final_collection))
    average_pairs_found.append(np.mean(pairs_found))


# Make a table of the results with pandas
df = pd.DataFrame({
    "c1": c1_factor_increases,
    "Success Rate": actual_success_rates,
    "Average Pairs Found": average_pairs_found,
    "Average Iterations": average_iterations,
    "Average Precritical Iterations": average_iterations_precritical,
    "Average Postcritical Iterations": average_iterations_postcritical,
    "Average Final Collection Iterations": average_iterations_final_collection
})


df    

## Experiment 5
* Modifying the lambda values

In [ ]:
n_particles = 100
n_neighbors = 30

nu = math.comb(n_particles, 2)          # nu is n choose 2
mu = n_particles * n_neighbors // 2     # mu is the number of close neighbor

alpha = 1
lambda_values_list = list(np.arange(5, 8, 0.5)/5)

actual_success_rates = []
average_iterations = []
average_iterations_precritical = []
average_iterations_postcritical = []
average_iterations_final_collection = []
average_pairs_found = []

runs = 1000


for i, lambda_value in enumerate(lambda_values_list):


    hyperparameters = CloseNeighborsHyperparameters(
        final_collection_k=lambda n, alpha, A1 : (alpha+1) * 4 * A1 * np.log(n),
        m_factor_increase=1,
        c1=450,
        c2=6,
        lamda_val=lambda_value
    )

    grover_iterations_precritical = []
    grover_iterations_postcritical = []
    grover_iterations_final_collection = []
    grover_iterations = []
    pairs_found = []
    successes = 0
    for _ in tqdm(range(runs)):
        n_found, iterations = close_neighbors(mu, alpha, hyperparameters, FastAbstractGrovers(nu, mu))
        pairs_found.append(n_found/mu)
        grover_iterations_precritical.append(iterations[0])
        grover_iterations_postcritical.append(iterations[1])
        grover_iterations_final_collection.append(iterations[2])
        grover_iterations.append(sum(iterations))
        if n_found == mu:
            successes += 1

    actual_success_rate = successes / runs
    actual_success_rates.append(actual_success_rate)
    average_iterations.append(np.mean(grover_iterations))
    average_iterations_precritical.append(np.mean(grover_iterations_precritical))
    average_iterations_postcritical.append(np.mean(grover_iterations_postcritical))
    average_iterations_final_collection.append(np.mean(grover_iterations_final_collection))
    average_pairs_found.append(np.mean(pairs_found))


# Make a table of the results with pandas
df = pd.DataFrame({
    "c1": lambda_values_list,
    "Success Rate": actual_success_rates,
    "Average Pairs Found": average_pairs_found,
    "Average Iterations": average_iterations,
    "Average Precritical Iterations": average_iterations_precritical,
    "Average Postcritical Iterations": average_iterations_postcritical,
    "Average Final Collection Iterations": average_iterations_final_collection
})


df  